In [1]:
from frameworks.LightenDiffusion.models.LightenDiffusion import Stage1
from dataset_registery.registery import DatasetManager
from frameworks.LightenDiffusion.models.decom import ImageEncoder, ImageDecoder, RetinexDecomposition
from eda.helpers.data_helpers import split_dataloader
from frameworks.LightenDiffusion.visualization.visualize_stage1 import visualize_stage1_results
from evaluation.lighten_diffusion_stage1 import evaluate_stage1_metrics
import os
import torch
import pandas as pd
from IPython.display import display

%load_ext autoreload
%autoreload 2

/home/grads/o/omarkhater/projects/lle-generative-priors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "SICE_paired"
dataset_id = "okhater/SICE"
source = "huggingface"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [ ]:
registry = DatasetManager()
registry.initialize_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "test"],
    dataset_type="paired",
    hf_cache_dir=f"../../datasets/{data_name}",
)

train_loader = registry.get_dataloader(data_name, "train", batch_size=16, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=16, shuffle=True)
train_loader, val_loader = split_dataloader(train_loader, split_ratio=0.2)

loaders = {
    "train": train_loader, 
    "val": val_loader, 
    "test": test_loader
    }
for name, loader in loaders.items():
    total_samples = len(loader.dataset)
    print(f"Loader: {name}, Total samples: {total_samples}")
    for low_imgs, label in loader:
        print(f"Batch contains {len(low_imgs)} low image samples.")
        print("Low images batch shape:", low_imgs.shape)
        print("Label batch shape:", label.shape)
        break
    print("==="*20)

In [ ]:
directory = "/home/grads/o/omarkhater/projects/lle-generative-priors/frameworks/LightenDiffusion/trained_models/stage1/"
file_path = os.path.join(directory, "best_stage1.pth")
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")
stage1_model_weights = torch.load(file_path, map_location=device)


stage1_model = Stage1(
    encoder=ImageEncoder(64),
    decoder=ImageDecoder(64),
    decomposer=RetinexDecomposition(),
)

stage1_model.load_state_dict(stage1_model_weights)

## Quantitive Evaluation

In [ ]:
metrics_train = evaluate_stage1_metrics(stage1_model,train_loader)

In [ ]:
metrics_test = evaluate_stage1_metrics(stage1_model,train_loader)

In [ ]:
results_df = pd.DataFrame({
    "train": metrics_train,
    "test": metrics_test
})
display(results_df)

## Qualitives Evaluation

In [ ]:
visualize_stage1_results(
    stage1_model,
    train_loader,
    num_samples = 8
)

In [ ]:
visualize_stage1_results(
    stage1_model,
    test_loader,
    num_samples = 8
)